# Microscopy color calibration
## Historical reproduction

Follow the original data from spectra and images to a gamut comparison and per-microscope polynomial calibration. Run cells in order, or choose **Restart Kernel and Run All Cells**.

**Read first:** [data setup](../docs/en/data.md), [hardware](../docs/en/hardware.md), [sources and preprocessing](../docs/en/preprocessing.md). Download the spectra/reference and image bundles, then set `data_root` in `config.json`. Code is shared with the [Russian notebook](historical_ru.ipynb); only the explanation and display language change.

This is the official research workflow for [the calibration-slide paper](https://doi.org/10.1007/s11018-022-02014-7) and [the polynomial-calibration paper](https://doi.org/10.1088/1742-6596/2192/1/012010). The inputs are the surviving research files. The published 99.4% gamut coverage cannot be recovered exactly from this available snapshot.

### What this version preserves

We retain the original numeric choices: row-index spectral pairing, glass OD normalization, float byte-scale RGB, quantized spectral-reference RGB, and 98 fitting rows including the illuminator. These reproduce the old computation; they are not recommendations for a new measurement. Complex OD and the nonstandard archived error aggregate are identified explicitly. For the alternative calculation, open [corrected_en.ipynb](corrected_en.ipynb).

## 1. Configuration and a compact input check

The helper checks all field folders and required inputs. The table summarizes what was found. Paths and checks stay outside the scientific cells.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "preprocessing").is_dir())
sys.path.insert(0, str(root))
from preprocessing.common import load_config
from analysis import figures, reporting
config = load_config()
PROFILE, LANGUAGE = "historical", "en"
figures.setup()
reporting.inventory(config, LANGUAGE)

## 2. Observer functions and illumination

Copy the archived tables without changing their wavelength grids. Gamut integrations use **D50 and the physiologically based 2° observer (`2_2006`)**. Calibration later uses the measured illuminator and `10_2006`. CIE 1931/D65 is used only for the colored diagram background. [Source and naming](../docs/en/observers.md).

In [ ]:
from preprocessing import observers
observers.main(config)
figures.plot_reference_functions(config, LANGUAGE)

## 3. Camera response curves from the original plot

The Sony ICX274AQ graph was manually separated into channel masks during the research. Here we repeat centerline extraction and calibrated pixel-to-wavelength conversion from those saved masks. These curves describe **relative sensor sensitivity**, not measured RGB colors of patches. [Source image, manual step and axis calibration](../docs/en/camera.md).

In [ ]:
from preprocessing import camera
camera.main(config)
figures.plot_camera(config, LANGUAGE)

## 4. Dye spectra

The 18 optical-density spectra were supplied by **Daria Dobrolyubova**, the author's former colleague and fellow BMSTU PhD student. Their measurement protocol is not recorded. Convert the stored wavelength/OD vectors to reusable CSV files and inspect examples. Transmission will be calculated as **T = 10^(−OD)**, without normalizing each dye's OD. [Provenance and format](../docs/en/dyes.md).

In [ ]:
from preprocessing import dyes
dyes.main(config)
figures.plot_dyes(config, LANGUAGE)

## 5. Other calibration targets

First extract the nine transmittance curves from **Bautista et al. (2014), Figure 5a**. The 0.1 nm output spacing is an interpolation grid, not the original measurement resolution. [Digitization details](../docs/en/bautista.md).

In [ ]:
from preprocessing import bautista
bautista.main(config)
figures.plot_bautista(config, LANGUAGE)

Read the remaining targets in their **native formats** before converting to XYZ. ColorChecker is Lab/D50; Kodak is XYZ on a 0–100 scale; IFRAO is Adobe RGB/D65 on a 0–255 scale; Korzynska contains eight Lab triplets with an inherited illuminant-A assumption. The table keeps those distinctions visible. These sources are not interchangeable spectral measurements. [Dataset descriptions](../docs/en/preprocessing.md).

In [ ]:
from preprocessing import reference_targets
from analysis.io import read_colors
target_folder = reference_targets.main(config, PROFILE)
target_tables = {p.stem: read_colors(p) for p in target_folder.glob("*.csv")}
reporting.targets(target_tables, LANGUAGE)

## 6. Raw measured spectra → transmission and optical density

For **both microscopes**, divide the available spectral exports by their recorded exposure, then use the measured illuminator to calculate T and OD. The note mentions 10 signal and 10 background frames, but the saved inputs are already instrument exports; we do not invent or repeat an undocumented background subtraction.

The following plots show selected Meiji fields. For the later gamut comparison, the glass transmission is combined with common D50. For calibration, each microscope retains its **own measured illumination**. [Processing and acquisition limitations](../docs/en/measured_fields.md).

In [ ]:
from preprocessing import measured_fields
measured_fields.main(config, PROFILE)
figures.plot_measured(config, PROFILE, LANGUAGE)
reporting.spectral_quality(config, PROFILE, LANGUAGE)

The quality table counts complex/invalid OD samples. Negative detector values can yield complex logarithms in this profile. The historical gamut calculation normalizes each glass OD by its maximum and takes real xy only **after** XYZ normalization. These are preserved computational choices, not physical optical densities.

## 7. Images → registered field colors

Average all pixels within each numbered frame, then average frame means equally. The first run reads 1,290 images; later runs reuse the cache. The spectral reference and registered Lab tables are paired by exact field names. **Camera exposure is not taken from the spectrometer exposure workbook.** [Image processing](../docs/en/images.md).

In [ ]:
from preprocessing import image_fields
image_fields.main(config, PROFILE)
registered = read_colors(config.artifacts / PROFILE / "meiji/registered_lab.csv")
reference = read_colors(config.artifacts / PROFILE / "meiji/reference_lab.csv")
import pandas as pd
pd.DataFrame({"L": registered.values[:6, 0], "a": registered.values[:6, 1],
              "b": registered.values[:6, 2]}, index=registered.names[:6])

## 8. All datasets in xy and the gamut figure

Integrate the dyes and glass transmission, load the prepared targets, and construct the CCD response triangle. **xy describes chromaticity only; brightness is removed.** The colored background is a display aid and is not used for the coverage calculation.

The static figure shows glass fields, their hull, the dye hull, Kodak Q60 and D50. The table uses the **dye hull as reference**, as in the paper's coverage comparison. Coverage is intersection area divided by reference-hull area; density is the number of fields divided by their own hull area.

In [ ]:
from analysis.workflow import calculate_gamuts, coverage_table
datasets, color_tables = calculate_gamuts(config, PROFILE)
figures.plot_paper_gamut(config, datasets, PROFILE, LANGUAGE)
coverage = coverage_table(datasets, reference="dyes")
reporting.table(coverage, LANGUAGE)

With the available files, glass coverage is **99.6716%**, whereas the paper reports **99.4%**. The older input snapshot has not been found; no input is adjusted to force the printed percentage. The exported XYZ/xy tables and figure make this available-data reproduction inspectable.

## 9. Polynomial correction within each microscope

Fit **registered Lab → spectral-reference Lab**, independently for Meiji and Olympus. Compare degrees 1–5 with and without cross terms. The plotted historical aggregate and standard mean ΔE76 are different quantities:

* Standard mean: average of `sqrt(ΔL² + Δa² + Δb²)` over fields.
* Archived aggregate: `(sqrt(sum(ΔL²)) + sqrt(sum(Δa²)) + sqrt(sum(Δb²))) / N`.

The archived aggregate reproduces the paper's rounded **0.75 and 0.66**, but must not be called mean ΔE76. [Experiment and metric definitions](../docs/en/calibration.md).

In [ ]:
from analysis.calibration import run_experiment
comparison, models, pairs = run_experiment(config, PROFILE)
figures.plot_calibration(config, comparison, PROFILE, LANGUAGE)
reporting.selected_models(comparison, LANGUAGE)

These models use 98 rows including the illuminator. Without cross terms, the archived implementation fits each output only from its matching input channel; this differs from the all-channel additive formula printed in the paper. Large historical Lab inputs also make high degrees numerically sensitive. Increasing training error is not sufficient evidence of overfitting.

Below is a **separate fit of the archived Lab tables** to check the printed rounded scores. It does not replace the spectra/image recomputation above. The final column quantifies recomputed versus saved registered-Lab differences; Olympus is not exactly identical.

In [ ]:
reporting.archived_comparison(config, PROFILE, LANGUAGE)

## 10. Apply the transform to a recorded image

For each microscope, apply its fitted model to every pixel of a recorded **СЗС-22** glass-field image. The third panel is the uniform spectral-reference color for display. There is no measured spatial ground-truth image here.

The table distinguishes the error of the field mean from mean pixelwise error. ΔE uses **unclipped Lab**, while the rendered RGB preview is clipped to the display range. A model fitted to field means is not automatically validated for every pixel or stained tissue.

In [ ]:
image_metrics = []
for system in ("meiji", "olympus"):
    metrics = figures.image_example(config, models[system], pairs[system],
                                    system, PROFILE, LANGUAGE)
    image_metrics.append(metrics)
reporting.save_image_metrics(config, PROFILE, image_metrics)
reporting.table(image_metrics, LANGUAGE)

## 11. Explore whole datasets interactively

Toggle **points** and **hulls** for complete datasets. Choose the reference hull; the default is the approximate CCD triangle. The table includes visible datasets and always includes the reference. Hiding datasets does not change any stored coordinates or the definition of a hull.

The same interactive figure is saved as `artifacts/<profile>/chromaticity_en.html`; it opens offline in a browser. Use **Save SVG** to export the current selection. If notebook scripting is blocked, trust this locally generated notebook or open the standalone HTML. Figures, model coefficients, per-field errors and the numerical environment remain beside it in the artifact folder.

In [ ]:
from analysis.interactive import create_explorer
reporting.save_environment(config, PROFILE)
create_explorer(config, datasets, PROFILE, LANGUAGE)